# Airbnb New User Bookings


<img src='https://images.surfacemag.com/app/uploads/2022/10/20115518/airbnb-logo.jpeg'>


Bu çalışmada `Airbnb New User Bookings` yarışması için kullanıcı profil bilgileri kullanılarak yeni kullanıcının ilk rezervasyon ülkesini tahmin eden bir başlangıç modeli oluşturulmuştur.


## Akış

1. Kütüphaneleri yükleme
2. Drive bağlama ve zip içeriğini tanımlama
3. Veri dosyalarını yükleme
4. Ön işleme
5. Özellik çıkarımı
6. Model kurma
7. Accuracy değerlendirmesi
8. Test tahmini ve submission
9. Sonuç


## 1. Kütüphaneleri Yükleme


In [1]:
# Bu bölümde kullanıcı ülke tahmini için gerekli veri hazırlama ve sınıflandırma kütüphanelerini içe aktarıyoruz.


In [2]:
from google.colab import drive
import os
import zipfile
import tempfile

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


## 2. Drive Bağlama ve Zip İçeriğini Tanımlama


In [3]:
# Bu bölümde yarışma zip dosyasını Drive üzerinden bağlıyor ve gerekli dosyaları buluyoruz.


In [4]:
drive.mount('/content/drive', force_remount=True)
zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/airbnb-recruiting-new-user-bookings.zip'
base_data_dir = '/content/drive/MyDrive/Colab Data Dosyaları'
print('Veri klasörü mevcut mu?:', os.path.exists(base_data_dir))
if os.path.exists(base_data_dir):
    print('Klasördeki ilk dosyalar:', os.listdir(base_data_dir)[:20])
print('Zip dosyası:', zip_path)
print('Zip mevcut mu?:', os.path.exists(zip_path))
if not os.path.exists(zip_path):
    raise FileNotFoundError('Zip dosyası bulunamadı. Colab Data Dosyaları içindeki gerçek dosya adını kontrol et.')
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_members = zip_ref.namelist()
print('Zip içindeki ilk dosyalar:', zip_members[:20])
def find_zip_member(members, filename):
    for member in members:
        if member.endswith(filename):
            return member
    raise FileNotFoundError(f'{filename} zip içinde bulunamadi.')
train_member = find_zip_member(zip_members, 'train_users_2.csv.zip')
test_member = find_zip_member(zip_members, 'test_users.csv.zip')
sample_submission_member = find_zip_member(zip_members, 'sample_submission_NDF.csv.zip')
print('Train member:', train_member)
print('Test member:', test_member)
print('Sample submission member:', sample_submission_member)


Mounted at /content/drive
Veri klasörü mevcut mu?: True
Klasördeki ilk dosyalar: ['Skin_Data.zip', 'Date_Fruit_Dataset.zip', 'Grape_Disease_Detection_Dataset.zip', 'GermanTrafficSign.zip', 'Malaria Data.zip', 'Electronics_5.json.zip', 'Fashion Product Images.zip', 'YouTube Analytics Data.zip', 'E-Commerce Marketing & Sales Revenue Prediction.zip', 'Customer_Spending_Dataset.zip', 'Predict Customer Purchase Behavior Dataset.zip', 'E-commerce Customer Churn Dataset.zip', 'Predict Conversion in Digital Marketing Dataset.zip', 'Customer Segmentation.zip', 'Customer Personality Analysis.zip', 'Wholesale Customers Data.zip', 'MNIST Handwritten Digits.zip', 'Face Mask Detection Dataset.zip', 'Emotion Detection.zip', 'Twitter Tweets Sentiment Dataset.zip']
Zip dosyası: /content/drive/MyDrive/Colab Data Dosyaları/airbnb-recruiting-new-user-bookings.zip
Zip mevcut mu?: True
Zip içindeki ilk dosyalar: ['age_gender_bkts.csv.zip', 'countries.csv.zip', 'sample_submission_NDF.csv.zip', 'sessions.csv.

## 3. Veri Dosyalarını Yükleme


In [5]:
# Bu bölümde iç zip dosyalarını açıp train, test ve submission tablolarını yüklüyoruz.


In [6]:
def read_csv_from_nested_zip(zip_path, member_name, **kwargs):
    with tempfile.TemporaryDirectory() as tmpdir:
        inner_zip_path = os.path.join(tmpdir, os.path.basename(member_name))

        with zipfile.ZipFile(zip_path, 'r') as outer_zip:
            with outer_zip.open(member_name) as src_file, open(inner_zip_path, 'wb') as dst_file:
                dst_file.write(src_file.read())

        with zipfile.ZipFile(inner_zip_path, 'r') as inner_zip:
            inner_members = inner_zip.namelist()
            csv_member = next((m for m in inner_members if m.endswith('.csv')), None)
            if csv_member is None:
                raise FileNotFoundError('İç zip içinde csv bulunamadı.')
            with inner_zip.open(csv_member) as f:
                return pd.read_csv(f, **kwargs)

train = read_csv_from_nested_zip(zip_path, train_member)
test = read_csv_from_nested_zip(zip_path, test_member)
sample_submission = read_csv_from_nested_zip(zip_path, sample_submission_member)

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Sample submission shape:', sample_submission.shape)
train.head()


Train shape: (213451, 16)
Test shape: (62096, 15)
Sample submission shape: (62096, 2)


,id,date_account_created,timestamp_first_active,date_first_booking,gender,age,signup_method,signup_flow,language,affiliate_channel,affiliate_provider,first_affiliate_tracked,signup_app,first_device_type,first_browser,country_destination
0,gxn3p5htnn,2010-06-28,20090319043255,NaN,-unknown-,NaN,facebook,0,en,direct,direct,untracked,Web,Mac Desktop,Chrome,NDF
1,820tgsjxq7,2011-05-25,20090523174809,NaN,MALE,38.0,facebook,0,en,seo,google,untracked,Web,Mac Desktop,Chrome,NDF
2,4ft3gnwmtx,2010-09-28,20090609231247,2010-08-02,FEMALE,56.0,basic,3,en,direct,direct,untracked,Web,Windows Desktop,IE,US
3,bjjt8pjhuk,2011-12-05,20091031060129,2012-09-08,FEMALE,42.0,facebook,0,en,direct,direct,untracked,Web,Mac Desktop,Firefox,other
4,87mebub9p4,2010-09-14,20091208061105,2010-02-18,-unknown-,41.0,basic,0,en,direct,direct,untracked,Web,Mac Desktop,Chrome,US


## 4. Ön İşleme


In [7]:
# Bu bölümde tarih alanlarını açıyor ve eksik bilgileri düzenliyoruz.


In [8]:
train_df = train.copy()
test_df = test.copy()

for df in [train_df, test_df]:
    df['date_account_created'] = pd.to_datetime(df['date_account_created'], errors='coerce')
    df['timestamp_first_active'] = pd.to_datetime(df['timestamp_first_active'].astype(str).str[:8], format='%Y%m%d', errors='coerce')

    df['dac_year'] = df['date_account_created'].dt.year
    df['dac_month'] = df['date_account_created'].dt.month
    df['dac_day'] = df['date_account_created'].dt.day
    df['tfa_year'] = df['timestamp_first_active'].dt.year
    df['tfa_month'] = df['timestamp_first_active'].dt.month
    df['tfa_day'] = df['timestamp_first_active'].dt.day

    for col in ['gender', 'signup_method', 'signup_flow', 'language', 'affiliate_channel', 'affiliate_provider', 'first_affiliate_tracked', 'signup_app', 'first_device_type', 'first_browser']:
        if col in df.columns:
            df[col] = df[col].fillna('missing').astype(str)

print('Train null count:', train_df.isnull().sum().sum())
print('Test null count:', test_df.isnull().sum().sum())
train_df.head()


Train null count: 212533
Test null count: 90972


,id,date_account_created,timestamp_first_active,date_first_booking,gender,age,signup_method,signup_flow,language,affiliate_channel,affiliate_provider,first_affiliate_tracked,signup_app,first_device_type,first_browser,country_destination,dac_year,dac_month,dac_day,tfa_year,tfa_month,tfa_day
0,gxn3p5htnn,2010-06-28,2009-03-19,NaN,-unknown-,NaN,facebook,0,en,direct,direct,untracked,Web,Mac Desktop,Chrome,NDF,2010,6,28,2009,3,19
1,820tgsjxq7,2011-05-25,2009-05-23,NaN,MALE,38.0,facebook,0,en,seo,google,untracked,Web,Mac Desktop,Chrome,NDF,2011,5,25,2009,5,23
2,4ft3gnwmtx,2010-09-28,2009-06-09,2010-08-02,FEMALE,56.0,basic,3,en,direct,direct,untracked,Web,Windows Desktop,IE,US,2010,9,28,2009,6,9
3,bjjt8pjhuk,2011-12-05,2009-10-31,2012-09-08,FEMALE,42.0,facebook,0,en,direct,direct,untracked,Web,Mac Desktop,Firefox,other,2011,12,5,2009,10,31
4,87mebub9p4,2010-09-14,2009-12-08,2010-02-18,-unknown-,41.0,basic,0,en,direct,direct,untracked,Web,Mac Desktop,Chrome,US,2010,9,14,2009,12,8


## 5. Özellik Çıkarımı


In [9]:
# Bu bölümde model girdilerini hazırlıyor ve kullanılacak alanları belirliyoruz.


In [10]:
feature_columns = [
    'gender', 'age', 'signup_method', 'signup_flow', 'language', 'affiliate_channel',
    'affiliate_provider', 'first_affiliate_tracked', 'signup_app', 'first_device_type',
    'first_browser', 'dac_year', 'dac_month', 'dac_day', 'tfa_year', 'tfa_month', 'tfa_day'
]

x = train_df[feature_columns].copy()
y = train_df['country_destination'].copy()
test_x = test_df[feature_columns].copy()

categorical_features = ['gender', 'signup_method', 'signup_flow', 'language', 'affiliate_channel', 'affiliate_provider', 'first_affiliate_tracked', 'signup_app', 'first_device_type', 'first_browser']
for col in categorical_features:
    x[col] = x[col].astype(str)
    test_x[col] = test_x[col].astype(str)

x_train, x_valid, y_train, y_valid = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

numerical_features = [col for col in x.columns if col not in categorical_features]

print('x_train shape:', x_train.shape)
print('x_valid shape:', x_valid.shape)


x_train shape: (170760, 17)
x_valid shape: (42691, 17)


## 6. Model Kurma


In [11]:
# Bu bölümde kullanıcı profil bilgileri ile ilk rezervasyon ülkesini tahmin eden başlangıç modelini kuruyoruz.


In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), numerical_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features)
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=180,
        max_depth=18,
        min_samples_split=8,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced_subsample'
    ))
])

model.fit(x_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['age', 'dac_year',
                                                   'dac_month', 'dac_day',
                                                   'tfa_year', 'tfa_month',
                                                   'tfa_day']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['gender', 'signup_method',
                                                   'signup_flow', 'language',
                                                   'affiliate_channel',
                                                   'affiliate_provider',
                                                   'first_affiliate_tracked',
                                                   'signup_app',
                                                   'first_device_type',
                                                   'first_browser'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced_subsample',
                                        max_depth=18, min_samples_split=8,
                                        n_estimators=180, n_jobs=-1,
                                        random_state=42))])

## 7. Accuracy Değerlendirmesi


In [13]:
# Bu bölümde modelin ilk rezervasyon ülkesini doğru tahmin etme oranını ölçüyoruz.


In [14]:
valid_preds = model.predict(x_valid)
accuracy = accuracy_score(y_valid, valid_preds)
print('Validation Accuracy:', round(accuracy, 5))


Validation Accuracy: 0.47497


## 8. Test Tahmini ve Submission


In [15]:
# Bu bölümde test kullanıcıları için ilk rezervasyon ülkesi tahminlerini oluşturuyoruz.


In [16]:
test_preds = model.predict(test_x)

submission = pd.DataFrame({
    'id': test_df['id'].values,
    'country': test_preds
})

print('Submission shape:', submission.shape)
submission.head()


Submission shape: (62096, 2)


,id,country
0,5uwns89zht,NDF
1,jtl0dijy2j,NDF
2,xx0ulgorjt,NDF
3,6c6puo6ix0,NDF
4,czqhjk3yfe,NDF


In [17]:
output_path = '/content/airbnb_submission.csv'
submission.to_csv(output_path, index=False)
print('Kaydedilen dosya:', output_path)


Kaydedilen dosya: /content/airbnb_submission.csv


## 9. Sonuç


Bu çalışmada Airbnb New User Bookings yarışması için kullanıcı profil ve kayıt bilgileri kullanılarak yeni kullanıcının ilk rezervasyon ülkesini tahmin eden bir başlangıç modeli oluşturuldu. Elde edilen sonuçlara göre model validation verisi üzerinde 0.47497 accuracy değeri elde etti ve test kullanıcıları için ilk rezervasyon ülke tahminleri üretildi.